In [1]:
# ============================================================
# VIDEO GAME SALES DATASET
# K-MEANS CLUSTERING + NLP
# Dataset:
# Video_Games_Sales_as_at_22_Dec_2016.csv
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "Video_Games_Sales_as_at_22_Dec_2016.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


# ============================================================
# 3. BASIC INFORMATION
# ============================================================

print("Dataset Information:")
print("=" * 60)

df.info()


# ============================================================
# 4. COLUMN NAMES
# ============================================================

print("\nColumn names:")
print("=" * 60)

for column in df.columns:
    print(column)


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing values:")
print("=" * 60)

missing_values = df.isnull().sum()

print(missing_values)

print("\nPercentage of missing values:")
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage.sort_values(ascending=False))


# ============================================================
# 6. CHECK DUPLICATES
# ============================================================

print("\nNumber of duplicate rows:", df.duplicated().sum())


# ============================================================
# 7. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)


# ============================================================
# 8. DATA TYPES
# ============================================================

print("\nData types:")
print(df.dtypes)


# ============================================================
# 9. CLEAN NUMERICAL COLUMNS
# ============================================================

numeric_columns = [
    "Year",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 10. CLEAN TEXT COLUMNS
# ============================================================

text_columns = [
    "Name",
    "Platform",
    "Genre",
    "Publisher"
]

for column in text_columns:
    df[column] = df[column].fillna("Unknown")


# ============================================================
# 11. HANDLE MISSING YEAR
# ============================================================

df["Year"] = df["Year"].fillna(
    df["Year"].median()
)


# ============================================================
# 12. HANDLE MISSING SALES VALUES
# ============================================================

sales_columns = [
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

for column in sales_columns:
    df[column] = df[column].fillna(0)


# ============================================================
# 13. REMOVE INVALID ROWS
# ============================================================

df = df.dropna(
    subset=["Global_Sales"]
)

print("Final cleaned shape:", df.shape)


# ============================================================
# 14. SUMMARY STATISTICS
# ============================================================

print("\nSummary statistics:")
display(df.describe())


# ============================================================
# 15. UNIQUE VALUES
# ============================================================

print("\nNumber of unique values:")

for column in df.columns:
    print(
        f"{column}: {df[column].nunique()}"
    )


# ============================================================
# 16. TOP SELLING VIDEO GAMES
# ============================================================

top_games = df.sort_values(
    by="Global_Sales",
    ascending=False
).head(20)

display(
    top_games[
        [
            "Name",
            "Platform",
            "Genre",
            "Year",
            "Global_Sales"
        ]
    ]
)


# ============================================================
# 17. TOP PUBLISHERS
# ============================================================

top_publishers = (
    df.groupby("Publisher")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))

top_publishers.sort_values().plot(
    kind="barh"
)

plt.title("Top 15 Publishers by Global Sales")
plt.xlabel("Global Sales (millions)")
plt.ylabel("Publisher")

plt.tight_layout()
plt.show()


# ============================================================
# 18. SALES BY GENRE
# ============================================================

genre_sales = (
    df.groupby("Genre")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))

genre_sales.plot(
    kind="bar"
)

plt.title("Global Sales by Genre")
plt.xlabel("Genre")
plt.ylabel("Global Sales (millions)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# 19. NUMBER OF GAMES BY GENRE
# ============================================================

plt.figure(figsize=(12, 6))

df["Genre"].value_counts().plot(
    kind="bar"
)

plt.title("Number of Games by Genre")
plt.xlabel("Genre")
plt.ylabel("Number of Games")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# 20. GLOBAL SALES DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["Global_Sales"],
    bins=50
)

plt.title("Distribution of Global Video Game Sales")
plt.xlabel("Global Sales (millions)")
plt.ylabel("Number of Games")

plt.tight_layout()
plt.show()


# ============================================================
# 21. SALES CORRELATION
# ============================================================

plt.figure(figsize=(10, 7))

correlation = df[
    sales_columns
].corr()

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f"
)

plt.title("Sales Correlation Matrix")

plt.tight_layout()
plt.show()


# ============================================================
# 22. PREPARE DATA FOR K-MEANS
# ============================================================

kmeans_features = [
    "Year",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

X = df[
    kmeans_features
].copy()

print("Features used for K-Means:")
print(kmeans_features)

print("\nFeature shape:", X.shape)


# ============================================================
# 23. STANDARDISE DATA
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Data standardised successfully.")


# ============================================================
# 24. ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(
        model.inertia_
    )


plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")

plt.xticks(k_values)

plt.tight_layout()
plt.show()


# ============================================================
# 25. SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)


plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.title("Silhouette Score")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")

plt.xticks(k_values)

plt.tight_layout()
plt.show()


# ============================================================
# 26. DISPLAY SILHOUETTE SCORES
# ============================================================

silhouette_table = pd.DataFrame({
    "K": list(k_values),
    "Silhouette_Score": silhouette_scores
})

display(
    silhouette_table
)


# ============================================================
# 27. SELECT NUMBER OF CLUSTERS
# ============================================================

# You can change this value after inspecting
# the Elbow Method and Silhouette Score.

best_k = 4

print("Selected number of clusters:", best_k)


# ============================================================
# 28. TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(
    X_scaled
)

print("K-Means clustering completed.")


# ============================================================
# 29. NUMBER OF GAMES IN EACH CLUSTER
# ============================================================

cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
)

print("Games in each cluster:")
print(cluster_counts)


plt.figure(figsize=(8, 5))

cluster_counts.plot(
    kind="bar"
)

plt.title("Number of Games in Each K-Means Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Games")

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


# ============================================================
# 30. CLUSTER SUMMARY
# ============================================================

cluster_summary = (
    df.groupby("Cluster")[
        kmeans_features
    ]
    .mean()
)

print("Average characteristics of each cluster:")

display(
    cluster_summary
)


# ============================================================
# 31. MEDIAN CLUSTER SUMMARY
# ============================================================

cluster_median = (
    df.groupby("Cluster")[
        kmeans_features
    ]
    .median()
)

print("Median characteristics of each cluster:")

display(
    cluster_median
)


# ============================================================
# 32. GENRE DISTRIBUTION BY CLUSTER
# ============================================================

genre_cluster = pd.crosstab(
    df["Cluster"],
    df["Genre"]
)

display(
    genre_cluster
)


# ============================================================
# 33. GENRE DISTRIBUTION HEATMAP
# ============================================================

plt.figure(figsize=(14, 7))

sns.heatmap(
    genre_cluster,
    annot=True,
    fmt="d"
)

plt.title(
    "Genre Distribution Across K-Means Clusters"
)

plt.xlabel("Genre")
plt.ylabel("Cluster")

plt.tight_layout()
plt.show()


# ============================================================
# 34. PLATFORM DISTRIBUTION BY CLUSTER
# ============================================================

platform_cluster = pd.crosstab(
    df["Cluster"],
    df["Platform"]
)

display(
    platform_cluster
)


# ============================================================
# 35. TOP GAMES IN EACH CLUSTER
# ============================================================

for cluster in sorted(
    df["Cluster"].unique()
):

    print("\n")
    print("=" * 70)
    print(f"CLUSTER {cluster}")
    print("=" * 70)

    cluster_games = (
        df[df["Cluster"] == cluster]
        .sort_values(
            "Global_Sales",
            ascending=False
        )
        .head(10)
    )

    display(
        cluster_games[
            [
                "Name",
                "Platform",
                "Genre",
                "Year",
                "Global_Sales"
            ]
        ]
    )


# ============================================================
# 36. PCA
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_scaled
)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

print(
    "PCA explained variance:"
)

print(
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)


# ============================================================
# 37. K-MEANS PCA VISUALISATION
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="Cluster",
    palette="Set2",
    alpha=0.6,
    s=50
)

plt.title(
    "K-Means Clusters Using PCA"
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.legend(
    title="Cluster"
)

plt.tight_layout()
plt.show()


# ============================================================
# 38. NLP SECTION
# ============================================================

print("=" * 70)
print("NLP ANALYSIS")
print("=" * 70)


# ============================================================
# 39. CREATE CLEAN GAME TITLE
# ============================================================

def clean_text(text):

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["Name_Clean"] = (
    df["Name"]
    .apply(clean_text)
)

display(
    df[
        ["Name", "Name_Clean"]
    ].head(20)
)


# ============================================================
# 40. TF-IDF
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = (
    tfidf_vectorizer.fit_transform(
        df["Name_Clean"]
    )
)

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)


# ============================================================
# 41. GET TF-IDF TERMS
# ============================================================

terms = (
    tfidf_vectorizer
    .get_feature_names_out()
)

print(
    "Number of NLP terms:",
    len(terms)
)


# ============================================================
# 42. FIND MOST IMPORTANT WORDS
# ============================================================

average_tfidf = np.asarray(
    tfidf_matrix.mean(axis=0)
).ravel()

tfidf_results = pd.DataFrame({
    "Term": terms,
    "TF_IDF": average_tfidf
})

tfidf_results = (
    tfidf_results
    .sort_values(
        "TF_IDF",
        ascending=False
    )
)

print(
    "Top 30 NLP terms:"
)

display(
    tfidf_results.head(30)
)


# ============================================================
# 43. VISUALISE TOP NLP TERMS
# ============================================================

top_terms = (
    tfidf_results
    .head(20)
    .sort_values("TF_IDF")
)

plt.figure(figsize=(12, 8))

plt.barh(
    top_terms["Term"],
    top_terms["TF_IDF"]
)

plt.title(
    "Top 20 Terms in Video Game Titles"
)

plt.xlabel(
    "Average TF-IDF Score"
)

plt.ylabel(
    "Term"
)

plt.tight_layout()
plt.show()


# ============================================================
# 44. NLP K-MEANS
# ============================================================

nlp_k = 5

nlp_kmeans = KMeans(
    n_clusters=nlp_k,
    random_state=42,
    n_init=10
)

df["NLP_Cluster"] = (
    nlp_kmeans.fit_predict(
        tfidf_matrix
    )
)

print(
    "NLP K-Means completed."
)

print(
    df["NLP_Cluster"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 45. NLP CLUSTER SILHOUETTE SCORE
# ============================================================

nlp_silhouette = silhouette_score(
    tfidf_matrix,
    df["NLP_Cluster"]
)

print(
    "NLP K-Means Silhouette Score:",
    round(nlp_silhouette, 4)
)


# ============================================================
# 46. IMPORTANT TERMS FOR EACH NLP CLUSTER
# ============================================================

print("=" * 70)
print("IMPORTANT TERMS BY NLP CLUSTER")
print("=" * 70)

for cluster in range(nlp_k):

    cluster_center = (
        nlp_kmeans
        .cluster_centers_[cluster]
    )

    top_indices = (
        cluster_center
        .argsort()[-15:][::-1]
    )

    top_words = terms[
        top_indices
    ]

    print(
        f"\nNLP Cluster {cluster}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 47. EXAMPLE GAMES FROM EACH NLP CLUSTER
# ============================================================

for cluster in sorted(
    df["NLP_Cluster"].unique()
):

    print("\n")
    print("=" * 70)
    print(f"NLP CLUSTER {cluster}")
    print("=" * 70)

    games = df[
        df["NLP_Cluster"] == cluster
    ][
        [
            "Name",
            "Platform",
            "Genre",
            "Global_Sales"
        ]
    ].head(15)

    display(games)


# ============================================================
# 48. NLP CLUSTER GENRE DISTRIBUTION
# ============================================================

nlp_genre = pd.crosstab(
    df["NLP_Cluster"],
    df["Genre"]
)

display(
    nlp_genre
)


# ============================================================
# 49. NLP GENRE HEATMAP
# ============================================================

plt.figure(figsize=(14, 7))

sns.heatmap(
    nlp_genre,
    annot=True,
    fmt="d"
)

plt.title(
    "Genres Across NLP Clusters"
)

plt.xlabel(
    "Genre"
)

plt.ylabel(
    "NLP Cluster"
)

plt.tight_layout()
plt.show()


# ============================================================
# 50. COMPARE NUMERICAL AND NLP CLUSTERS
# ============================================================

cluster_comparison = pd.crosstab(
    df["Cluster"],
    df["NLP_Cluster"]
)

print(
    "Numerical K-Means vs NLP K-Means:"
)

display(
    cluster_comparison
)


# ============================================================
# 51. CLUSTER COMPARISON HEATMAP
# ============================================================

plt.figure(figsize=(10, 7))

sns.heatmap(
    cluster_comparison,
    annot=True,
    fmt="d"
)

plt.title(
    "Comparison of Numerical and NLP Clusters"
)

plt.xlabel(
    "NLP Cluster"
)

plt.ylabel(
    "Sales/Year Cluster"
)

plt.tight_layout()
plt.show()


# ============================================================
# 52. GLOBAL SALES BY K-MEANS CLUSTER
# ============================================================

sales_by_cluster = (
    df.groupby("Cluster")
    ["Global_Sales"]
    .sum()
    .sort_values(
        ascending=False
    )
)

print(
    "Total global sales by cluster:"
)

display(
    sales_by_cluster
)


plt.figure(figsize=(10, 6))

sales_by_cluster.plot(
    kind="bar"
)

plt.title(
    "Total Global Sales by K-Means Cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Global Sales (millions)"
)

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


# ============================================================
# 53. AVERAGE GLOBAL SALES BY GENRE
# ============================================================

average_genre_sales = (
    df.groupby("Genre")
    ["Global_Sales"]
    .mean()
    .sort_values(
        ascending=False
    )
)

display(
    average_genre_sales
)


# ============================================================
# 54. SALES OVER TIME
# ============================================================

year_sales = (
    df.groupby("Year")
    ["Global_Sales"]
    .sum()
)

plt.figure(figsize=(14, 6))

plt.plot(
    year_sales.index,
    year_sales.values
)

plt.title(
    "Global Video Game Sales by Year"
)

plt.xlabel(
    "Year"
)

plt.ylabel(
    "Global Sales (millions)"
)

plt.tight_layout()
plt.show()


# ============================================================
# 55. TOP 20 GAMES OVERALL
# ============================================================

top_20 = (
    df.sort_values(
        "Global_Sales",
        ascending=False
    )
    .head(20)
)

display(
    top_20[
        [
            "Name",
            "Platform",
            "Genre",
            "Publisher",
            "Year",
            "Global_Sales",
            "Cluster",
            "NLP_Cluster"
        ]
    ]
)


# ============================================================
# 56. CREATE FINAL DATASET
# ============================================================

final_columns = [
    "Name",
    "Platform",
    "Year",
    "Genre",
    "Publisher",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales",
    "Cluster",
    "NLP_Cluster",
    "PCA1",
    "PCA2"
]

final_df = df[
    final_columns
].copy()

display(
    final_df.head(20)
)


# ============================================================
# 57. SAVE RESULTS
# ============================================================

output_file = (
    "Video_Games_Sales_KMeans_NLP_Results.csv"
)

final_df.to_csv(
    output_file,
    index=False
)

print(
    f"Results saved successfully to: {output_file}"
)


# ============================================================
# 58. SAVE CLUSTER SUMMARY
# ============================================================

cluster_summary.to_csv(
    "KMeans_Cluster_Summary.csv"
)

tfidf_results.to_csv(
    "NLP_TFIDF_Terms.csv",
    index=False
)

print(
    "Cluster summary and NLP results saved."
)


# ============================================================
# 59. FINAL RESULTS
# ============================================================

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "\nOriginal dataset shape:",
    df.shape
)

print(
    "\nNumber of K-Means clusters:",
    best_k
)

print(
    "Number of NLP clusters:",
    nlp_k
)

print(
    "\nK-Means silhouette score:",
    round(
        silhouette_score(
            X_scaled,
            df["Cluster"]
        ),
        4
    )
)

print(
    "NLP silhouette score:",
    round(
        nlp_silhouette,
        4
    )
)

print(
    "\nK-Means cluster sizes:"
)

print(
    df["Cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nNLP cluster sizes:"
)

print(
    df["NLP_Cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nAnalysis completed successfully!"
)

Libraries imported successfully.
Dataset loaded successfully!
Shape: (16719, 16)


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  str    
 1   Platform         16719 non-null  str    
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  str    
 4   Publisher        16665 non-null  str    
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  str    
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  str    
 15  Rating           9950 non-null   str    
dtypes: float64(9), str(7)
memory usage: 3.0 MB

Colu

KeyError: 'Year'